# Imputación y normalización de series producto × cliente + LightGBM
**Labo3 Rosario 2026** — predecir `tn` de 202002 (t+2) para los 780 productos de `product_id_apredecir201912.txt`.

Este notebook implementa, en orden:
1. **Distinción de ceros** usando la *ventana de vida* del producto (a nivel empresa) refinada por el alta del cliente → ceros reales = `0`, ceros estructurales = `NaN`.
2. **Neteo de negativos** por producto×cliente×período (neto<0 ⇒ 0; el faltante es `NaN`, nunca un negativo).
3. **Features de contexto**: `antiguedad_producto`, `antiguedad_cliente`, `meses_desde_inicio_serie`.
4. **Normalización por serie** (z-norm con `nanmean`/`nanstd`), guardando `mu`/`sd`/`inicio_serie` para invertir; manejo de series planas (`sd==0`).
5. **Visualizaciones**: (a) productos con serie incompleta y mes de lanzamiento; (b) formas que por escala parecían distintas y tras escalar son similares (similitud de coseno + ejemplo de múltiplos).
6. **Clustering por forma** (tslearn `KShape` sobre series z-norm alineadas por edad) → `cluster_id` como feature.
7. **LightGBM** a nivel **producto×cliente** con NaN nativos en lags, variables escaladas, máximo histórico y **racha de ceros del cliente**; optimización breve con Optuna; predicción agregada a producto y **submit a Kaggle**.

> **Decisión de diseño:** el modelo se entrena a nivel `product_id × customer_id` (target `tn` a t+2) y se **suma a `product_id`** para la submission. Así la *racha de 0s del cliente* es una feature natural. El clustering de formas se hace a nivel producto (serie agregada) y se propaga como `cluster_id`.


## 0. Init ambiente Google Colab
Misma mecánica que los notebooks de la cátedra: monta Drive, descarga datos y configura Kaggle.

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

In [ ]:
!pip install uv
!uv pip install -q kaggle lightgbm optuna tslearn

In [ ]:
import os

def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

PARAM = {
    'experimento': 'lgbm_imputnorm_01',
    'kaggle_competition': 'labo-iii-2026-rosario',
    'semilla': 102191,
}
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)
print('workdir:', ruta)

## 1. Carga de datos y armado del panel
`sell-in` viene a nivel período × cliente × producto. Lo agregamos a `product_id × customer_id × periodo`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = '/content/datasets'   # en local, ajustar a la carpeta de datasets

sell      = pd.read_csv(f'{DATA}/sell-in.txt.gz', sep='\t')
productos = pd.read_csv(f'{DATA}/tb_productos.txt', sep='\t')
stocks    = pd.read_csv(f'{DATA}/tb_stocks.txt', sep='\t')
apredecir = pd.read_csv(f'{DATA}/product_id_apredecir201912.txt', sep='\t')

print('sell-in   :', sell.shape, sell.columns.tolist())
print('productos :', productos.shape, productos.columns.tolist())
print('stocks    :', stocks.shape, stocks.columns.tolist())
print('apredecir :', apredecir.shape, apredecir.columns.tolist())
sell.head()

In [ ]:
# Helpers de período (YYYYMM int) <-> índice mensual continuo
def periodo_to_idx(p):
    p = int(p)
    return (p // 100) * 12 + (p % 100) - 1

def idx_to_periodo(i):
    i = int(i)
    return (i // 12) * 100 + (i % 12) + 1

pmin, pmax = int(sell['periodo'].min()), int(sell['periodo'].max())
GSTART, GEND = periodo_to_idx(pmin), periodo_to_idx(pmax)
all_periodos = [idx_to_periodo(i) for i in range(GSTART, GEND + 1)]
print(f'periodos: {pmin}..{pmax}  ({len(all_periodos)} meses)')
print('primeros/ultimos:', all_periodos[:3], all_periodos[-3:])

## 2. Neteo de negativos
Los negativos (devoluciones, NC, anulaciones) **no** son demanda negativa. Se netean por producto×cliente×período; si el neto queda `< 0` se lleva a `0`. El faltante **nunca** se codifica con negativos (eso es `NaN`).

In [ ]:
base = (sell.groupby(['product_id', 'customer_id', 'periodo'], as_index=False)['tn']
            .sum())                      # netea positivos y negativos del mismo p×c×periodo
n_neg = int((base['tn'] < 0).sum())
print(f'pares p×c×periodo con neto<0 (se llevan a 0): {n_neg}')
base['tn'] = base['tn'].clip(lower=0)     # neto<0 -> 0
base['pidx'] = base['periodo'].map(periodo_to_idx)

## 3. Ventana de existencia: ceros reales vs estructurales
- **Producto** (a nivel empresa): `[prod_start, prod_end]` = primer y último período en que *cualquier* cliente compró el producto (`tn>0`). NO usamos la regla "máximo de la empresa = 0 ⇒ no existía" período a período.
- **Cliente**: `cust_alta` = primer período con cualquier compra del cliente.
- **Inicio válido de cada serie**: `inicio_serie = max(prod_start, cust_alta)`.

Dentro de la serie válida y sin compra ⇒ **cero real (0)**. Fuera (producto no lanzado/discontinuado, o cliente no dado de alta aún) ⇒ **estructural (NaN)**.

In [ ]:
ventas_pos = base[base['tn'] > 0]

prod_win  = ventas_pos.groupby('product_id')['pidx'].agg(prod_start='min', prod_end='max')
cust_alta = ventas_pos.groupby('customer_id')['pidx'].min().rename('cust_alta')

print('productos con ventana definida:', prod_win.shape[0])
print('clientes con alta definida    :', cust_alta.shape[0])
prod_win.head()

In [ ]:
# Pares (producto, cliente) que alguna vez tuvieron compra del producto
pares = ventas_pos[['product_id', 'customer_id']].drop_duplicates().reset_index(drop=True)
pares = pares.merge(prod_win, on='product_id').merge(cust_alta, on='customer_id')
pares['inicio_serie'] = pares[['prod_start', 'cust_alta']].max(axis=1)
print('series (producto×cliente):', len(pares))

# Expansion VECTORIZADA de cada par a su rango [prod_start, prod_end]
lengths = (pares['prod_end'] - pares['prod_start'] + 1).to_numpy()
rep     = np.repeat(np.arange(len(pares)), lengths)
offsets = np.concatenate([np.arange(l) for l in lengths])

grid = pares.iloc[rep].reset_index(drop=True)
grid['pidx'] = grid['prod_start'].to_numpy() + offsets
print('filas del panel (con tramo estructural pre-alta):', len(grid))

In [ ]:
# Traigo la venta real neteada
grid = grid.merge(base[['product_id', 'customer_id', 'pidx', 'tn']],
                  on=['product_id', 'customer_id', 'pidx'], how='left')

estructural = grid['pidx'] < grid['inicio_serie']           # producto existe pero cliente no dado de alta
real_zero   = (~estructural) & grid['tn'].isna()            # dentro de serie valida y sin compra

grid.loc[real_zero, 'tn']   = 0.0                            # cero real
grid.loc[estructural, 'tn'] = np.nan                         # estructural -> NaN
grid['periodo'] = grid['pidx'].map(idx_to_periodo)

print('ceros reales imputados :', int(real_zero.sum()))
print('estructurales (NaN)    :', int(estructural.sum()))
print('positivos observados   :', int((grid["tn"] > 0).sum()))

## 4. Features de contexto
`antiguedad_producto`, `antiguedad_cliente` y `meses_desde_inicio_serie` (negativo en el tramo estructural pre-alta, que luego se descarta).

In [ ]:
grid['antiguedad_producto']      = grid['pidx'] - grid['prod_start']
grid['antiguedad_cliente']       = grid['pidx'] - grid['cust_alta']
grid['meses_desde_inicio_serie'] = grid['pidx'] - grid['inicio_serie']
grid[['product_id','customer_id','periodo','tn',
      'antiguedad_producto','antiguedad_cliente','meses_desde_inicio_serie']].head(8)

### Salida 1 — Panel con ceros reales = 0 y estructurales = NaN

In [ ]:
panel = grid[['product_id', 'customer_id', 'periodo', 'pidx', 'tn',
              'antiguedad_producto', 'antiguedad_cliente', 'meses_desde_inicio_serie']].copy()
panel.to_parquet('panel_ceros.parquet', index=False)
print('panel:', panel.shape, '| NaN estructurales:', int(panel["tn"].isna().sum()))
panel.head()

## 5. Normalización por serie (z-norm)
`z = (x - nanmean(x)) / nanstd(x)`, ignorando NaN pero **incluyendo los 0 reales**. Guardamos `mu`/`sd`/`inicio_serie` para invertir (`x = z*sd + mu`). Si `sd == 0` (serie constante o toda en 0) → grupo **plano**, `z = 0` (no dividimos).

In [ ]:
g = grid.groupby(['product_id', 'customer_id'])['tn']
norm = g.agg(mu='mean', n_obs='count').reset_index()        # mean/count ignoran NaN
sd   = g.std(ddof=0).rename('sd').reset_index()             # nanstd poblacional (ddof=0)
norm = norm.merge(sd, on=['product_id', 'customer_id'])
norm = norm.merge(pares[['product_id', 'customer_id', 'inicio_serie']],
                  on=['product_id', 'customer_id'], how='left')
norm['inicio_serie_periodo'] = norm['inicio_serie'].map(idx_to_periodo)
norm['flag_plano'] = norm['sd'].fillna(0).eq(0)

print('series planas (sd==0):', int(norm['flag_plano'].sum()), '/', len(norm))
norm.head()

### Salida 2 — Tabla de parámetros de normalización por serie

In [ ]:
tabla_norm = norm[['product_id', 'customer_id', 'mu', 'sd',
                   'inicio_serie_periodo', 'n_obs', 'flag_plano']]
tabla_norm.to_csv('parametros_normalizacion.csv', index=False)
tabla_norm.head()

In [ ]:
# Columna z en el panel (NaN se preservan; series planas -> z=0)
grid = grid.merge(norm[['product_id', 'customer_id', 'mu', 'sd', 'flag_plano']],
                  on=['product_id', 'customer_id'], how='left')
grid['z'] = np.where(grid['flag_plano'], 0.0, (grid['tn'] - grid['mu']) / grid['sd'])
grid.loc[grid['tn'].isna(), 'z'] = np.nan      # estructural se queda NaN tambien en z

## 6. Series alineadas por *edad* y z-normalizadas (para clustering por forma)
Para comparar **formas** comparables, alineamos por edad de la serie (meses desde el inicio del producto), no por fecha calendario. El clustering se hace a nivel **producto** (serie agregada sobre clientes).

In [ ]:
# Serie de producto = suma sobre clientes (min_count=1 conserva NaN si todo es NaN)
prod_period = (grid.groupby(['product_id', 'pidx'])['tn']
                   .sum(min_count=1).reset_index())
prod_period = prod_period.merge(prod_win, on='product_id')
prod_period['age'] = prod_period['pidx'] - prod_period['prod_start']

mat = prod_period.pivot(index='product_id', columns='age', values='tn').sort_index()

mu_p = mat.mean(axis=1)                      # nanmean por fila
sd_p = mat.std(axis=1, ddof=0)               # nanstd por fila
matz = mat.sub(mu_p, axis=0).div(sd_p.replace(0, np.nan), axis=0)   # z por producto (forma)
print('matriz de formas (productos x edad):', mat.shape)

## 7. Visualización A — Productos con serie incompleta y mes de lanzamiento
"Incompleta" = el producto **no** abarca todo el horizonte: se lanzó después del primer período (`prod_start > inicio global`) y/o se discontinuó antes del último (`prod_end < fin global`).

In [ ]:
pw = prod_win.reset_index()
pw['lanzamiento']        = pw['prod_start'].map(idx_to_periodo)
pw['ultimo_periodo']     = pw['prod_end'].map(idx_to_periodo)
pw['incompleta_inicio']  = pw['prod_start'] > GSTART
pw['incompleta_fin']     = pw['prod_end']   < GEND
pw['largo_meses']        = pw['prod_end'] - pw['prod_start'] + 1
incompletas = pw[pw['incompleta_inicio'] | pw['incompleta_fin']].copy()

print(f'productos totales      : {len(pw)}')
print(f'series incompletas     : {len(incompletas)}')
print(f'  lanzados despues      : {int(pw["incompleta_inicio"].sum())}')
print(f'  discontinuados antes  : {int(pw["incompleta_fin"].sum())}')
incompletas.sort_values('lanzamiento').head(10)[
    ['product_id','lanzamiento','ultimo_periodo','largo_meses','incompleta_inicio','incompleta_fin']]

In [ ]:
# Histograma: ¿en qué mes se empezaron a vender los productos lanzados despues del inicio?
lanzados = pw[pw['incompleta_inicio']].copy()
fig, ax = plt.subplots(1, 2, figsize=(15, 4))

orden = sorted(lanzados['lanzamiento'].unique())
counts = lanzados['lanzamiento'].value_counts().reindex(orden).fillna(0)
ax[0].bar(range(len(orden)), counts.values)
ax[0].set_xticks(range(len(orden)))
ax[0].set_xticklabels([str(p) for p in orden], rotation=90, fontsize=7)
ax[0].set_title('Mes de lanzamiento (productos que arrancan después del inicio)')
ax[0].set_ylabel('cantidad de productos')

ax[1].hist(pw['largo_meses'], bins=range(1, len(all_periodos) + 2), edgecolor='white')
ax[1].set_title('Largo de la ventana de vida del producto (meses)')
ax[1].set_xlabel('meses activos')
plt.tight_layout(); plt.show()

In [ ]:
# Timeline (Gantt) de una muestra de series incompletas
muestra = incompletas.sort_values('lanzamiento').head(35)
fig, ax = plt.subplots(figsize=(13, 9))
for k, (_, r) in enumerate(muestra.iterrows()):
    ax.barh(k, r['prod_end'] - r['prod_start'] + 1, left=r['prod_start'], height=0.6)
ax.set_yticks(range(len(muestra)))
ax.set_yticklabels(muestra['product_id'].astype(str), fontsize=7)
xt = list(range(GSTART, GEND + 1, 3))
ax.set_xticks(xt); ax.set_xticklabels([idx_to_periodo(i) for i in xt], rotation=90, fontsize=7)
ax.axvline(GSTART, color='k', ls='--', lw=0.7); ax.axvline(GEND, color='k', ls='--', lw=0.7)
ax.set_title('Ventana de vida de series incompletas (muestra)'); plt.tight_layout(); plt.show()

## 8. Visualización B — Formas similares tras escalar (similitud de coseno)
Antes de escalar, dos productos pueden verse muy distintos solo por **magnitud**. La similitud de coseno es **invariante a la escala**: si una serie es múltiplo de otra (`y = k·x`, k>0), su coseno es exactamente 1.

Buscamos pares con **coseno alto sobre las formas z-normalizadas** pero **escala (tamaño) muy distinta**, y graficamos antes/después.

In [ ]:
# Ilustración del concepto: el coseno NO distingue múltiplos; la distancia euclídea SÍ
a = np.array([0.20, 0.30, 0.25, 0.40, 0.35, 0.50])
for k, lbl in [(1.0, 'x'), (1.9, '1.9·x'), (5.0, '5·x')]:
    b = k * a
    cos = (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))
    print(f'coseno(a, {lbl:5}) = {cos:.4f}   | euclídea = {np.linalg.norm(a-b):.3f}')

# vs. una forma distinta (no múltiplo)
c = np.array([0.20, 0.38, 0.21, 0.55, 0.30, 0.60])
print(f'coseno(a, forma distinta) = {(a@c)/(np.linalg.norm(a)*np.linalg.norm(c)):.4f}')

In [ ]:
from numpy.linalg import norm as l2

# Matriz de similitud de coseno sobre las FORMAS z-normalizadas y alineadas por edad.
# Rellenamos NaN con 0 SOLO para este cálculo de forma (no afecta al modelo).
M  = matz.fillna(0.0).to_numpy()
Mn = M / (l2(M, axis=1, keepdims=True) + 1e-9)
S  = Mn @ Mn.T
prods = matz.index.to_numpy()

escala = mu_p.reindex(matz.index).to_numpy()            # tamaño (mu) por producto
iu = np.triu_indices(len(prods), k=1)
cos_vals   = S[iu]
ratio_esc  = np.maximum(escala[iu[0]], escala[iu[1]]) / (np.minimum(escala[iu[0]], escala[iu[1]]) + 1e-9)

cand = pd.DataFrame({'i': iu[0], 'j': iu[1], 'cos': cos_vals, 'ratio_escala': ratio_esc})
cand = cand[(cand['cos'] > 0.95) & (cand['ratio_escala'] > 3)].sort_values('cos', ascending=False)
print('pares con forma casi idéntica pero escala muy distinta:', len(cand))
cand.head(8)

In [ ]:
# Graficar uno de esos pares: crudo (escalas distintas) vs z-norm (formas se solapan)
if len(cand):
    r = cand.iloc[0]
    pi, pj = prods[int(r['i'])], prods[int(r['j'])]
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(mat.loc[pi].values,  label=f'{pi}', marker='.')
    ax[0].plot(mat.loc[pj].values,  label=f'{pj}', marker='.')
    ax[0].set_title(f'CRUDO (escala distinta, ratio≈{r["ratio_escala"]:.1f})'); ax[0].legend()
    ax[1].plot(matz.loc[pi].values, label=f'{pi}', marker='.')
    ax[1].plot(matz.loc[pj].values, label=f'{pj}', marker='.')
    ax[1].set_title(f'Z-NORM (forma similar, coseno={r["cos"]:.3f})'); ax[1].legend()
    plt.tight_layout(); plt.show()
else:
    print('No se hallaron pares con el umbral elegido; bajá cos o ratio_escala.')

## 9. Clustering por forma con tslearn `KShape`
Escalamos con `TimeSeriesScalerMeanVariance` (z por serie) y agrupamos formas con `KShape`. Las series planas (`sd_p==0`) van a un cluster aparte (`-1`). El `cluster_id` se usa después como feature.

In [ ]:
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import KShape

planos_mask = sd_p.reindex(matz.index).fillna(0).eq(0).to_numpy()
X_shape = matz.fillna(0.0).to_numpy()[~planos_mask]
prods_shape = prods[~planos_mask]

N_CLUSTERS = 6
cluster_map = {}
if len(X_shape) >= N_CLUSTERS:
    Xs = TimeSeriesScalerMeanVariance().fit_transform(X_shape[:, :, None])
    ks = KShape(n_clusters=N_CLUSTERS, random_state=PARAM['semilla'], max_iter=30)
    labels = ks.fit_predict(Xs)
    cluster_map = dict(zip(prods_shape, labels))
cluster_map.update({p: -1 for p in prods[planos_mask]})       # planos -> -1

prod_cluster = pd.DataFrame({'product_id': list(cluster_map.keys()),
                             'cluster_id': list(cluster_map.values())})
print(prod_cluster['cluster_id'].value_counts().sort_index())

In [ ]:
# Centroides / formas medias por cluster
clusters_validos = sorted([c for c in prod_cluster['cluster_id'].unique() if c >= 0])
if clusters_validos:
    ncol = 3; nrow = int(np.ceil(len(clusters_validos) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, clusters_validos):
        miembros = prod_cluster.loc[prod_cluster['cluster_id'] == c, 'product_id']
        sub = matz.reindex(miembros).to_numpy()
        ax.plot(np.nanmean(sub, axis=0), color='crimson', lw=2)
        ax.set_title(f'cluster {c}  (n={len(miembros)})')
    for ax in axes[len(clusters_validos):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 10. Dataset para LightGBM (producto × cliente)
- Quitamos las filas **estructurales** (`meses_desde_inicio_serie < 0`, que quedaron NaN).
- Lags y medias móviles con **NaN explícito** al inicio de serie (NO `fillna(0)`; "no hay historia" ≠ "venta 0").
- Variables **escaladas** (`z` y sus lags).
- **Máximo histórico** (expanding, leak-free).
- **Racha de ceros del cliente**: meses consecutivos sin compra hasta `t-1`.
- `cluster_id`, atributos de producto y stock.
- **Target** = `tn` a `t+2` (`shift(-2)` dentro de cada serie).

In [ ]:
df = grid[grid['meses_desde_inicio_serie'] >= 0].copy()        # fuera el tramo estructural
df = df.sort_values(['product_id', 'customer_id', 'pidx']).reset_index(drop=True)
gkey = ['product_id', 'customer_id']

gtn = df.groupby(gkey, sort=False)['tn']
for l in [1, 2, 3, 6, 12]:
    df[f'lag{l}'] = gtn.shift(l)                                # NaN al inicio: se preservan

s1 = gtn.shift(1)
df['_s1'] = s1
for w in [3, 6, 12]:
    df[f'rmean{w}'] = (df.groupby(gkey)['_s1']
                         .transform(lambda s: s.rolling(w, min_periods=2).mean()))
    df[f'rstd{w}']  = (df.groupby(gkey)['_s1']
                         .transform(lambda s: s.rolling(w, min_periods=2).std()))

df['tn_max_hist'] = df.groupby(gkey)['_s1'].cummax()           # máximo histórico (sin el actual)

# variables escaladas (z) y sus lags
gz = df.groupby(gkey, sort=False)['z']
for l in [1, 2, 3]:
    df[f'zlag{l}'] = gz.shift(l)

In [ ]:
# Racha de ceros del cliente: meses consecutivos con tn==0 hasta t-1 (vectorizado, leak-free).
# df ya está ordenado por serie; el shift(1) dentro de cada grupo da NaN->False en el inicio,
# por lo que la racha se reinicia en cada cambio de serie.
past_zero = (df.groupby(gkey)['tn'].shift(1) == 0).fillna(False).astype(int)
reset = past_zero.eq(0)
grp_runs = reset.cumsum()
df['racha_ceros'] = past_zero.groupby(grp_runs).cumsum().to_numpy()
print('racha_ceros: max', int(df['racha_ceros'].max()), '| media', round(df['racha_ceros'].mean(), 3))

In [ ]:
# Atributos de producto, stock y cluster
prod_cols = [c for c in productos.columns if c != 'product_id']
df = df.merge(productos, on='product_id', how='left')
df = df.merge(stocks, on=['product_id', 'periodo'], how='left')
df = df.merge(prod_cluster, on='product_id', how='left')
df['cluster_id'] = df['cluster_id'].fillna(-1).astype(int)

# Target a t+2
df['target'] = df.groupby(gkey)['tn'].shift(-2)
df.drop(columns=['_s1'], inplace=True)
print('df modelo:', df.shape)

In [ ]:
# Features categóricas (LightGBM las maneja nativamente como 'category')
cat_features = [c for c in ['cat1', 'cat2', 'cat3', 'brand', 'cluster_id'] if c in df.columns]
for c in cat_features:
    df[c] = df[c].astype('category')

drop_cols = {'target', 'tn', 'z', 'periodo', 'flag_plano'}
features = [c for c in df.columns if c not in drop_cols]
print(f'{len(features)} features')
print('categóricas:', cat_features)
print(features)

## 11. Splits temporales y métrica
- **Predicción** (202002): filas con `periodo == 201912`.
- **Train**: filas con `target` conocido. La última `periodo` entrenable es 201910 (target 201912).
- **Validación temporal**: `periodo ∈ {201909, 201910}` (targets 201911, 201912).

Métrica de la competencia (a nivel producto): `sum_p |pred_p − real_p| / sum_p real_p`. Para validar, agregamos las predicciones de cada par a `product_id` y comparamos contra la `tn` real de ese período objetivo.

In [ ]:
# tn real por producto×periodo (incluye ceros reales) para evaluar a nivel producto
real_prod = grid.groupby(['product_id', 'pidx'])['tn'].sum(min_count=1).reset_index()
real_prod['periodo_obj'] = real_prod['pidx'].map(idx_to_periodo)

def metrica_producto(df_eval, y_pred, periodo_objetivo):
    """df_eval: filas con product_id y pidx de la FEATURE; target real a t+2."""
    tmp = df_eval[['product_id']].copy()
    tmp['pred'] = np.clip(y_pred, 0, None)
    pred_p = tmp.groupby('product_id')['pred'].sum()
    real_p = (real_prod[real_prod['periodo_obj'] == periodo_objetivo]
              .set_index('product_id')['tn'])
    idx = pred_p.index.union(real_p.index)
    p = pred_p.reindex(idx).fillna(0); r = real_p.reindex(idx).fillna(0)
    return float(np.abs(p - r).sum() / (r.sum() + 1e-9))

PRED_PERIODO   = 201912        # feature -> predice 202002
VALID_PERIODOS = [201909, 201910]

predict_mask = df['periodo'] == PRED_PERIODO
train_mask   = df['target'].notna() & (~df['periodo'].isin(VALID_PERIODOS))
valid_mask   = df['target'].notna() & (df['periodo'].isin(VALID_PERIODOS))

print('train:', int(train_mask.sum()), '| valid:', int(valid_mask.sum()), '| predict:', int(predict_mask.sum()))

## 12. Optimización breve con Optuna
Pocos trials sobre los hiperparámetros principales de LightGBM, optimizando la métrica de producto en validación (target período 201910 ⇒ objetivo 201912).

In [ ]:
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

Xtr, ytr = df.loc[train_mask, features], df.loc[train_mask, 'target']
Xva, yva = df.loc[valid_mask, features], df.loc[valid_mask, 'target']
df_va_201910 = df.loc[valid_mask & (df['periodo'] == 201910)]
Xva_eval = df_va_201910[features]

def objective(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'l1',
        'verbosity': -1,
        'seed': PARAM['semilla'],
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq':      1,
        'lambda_l1':         trial.suggest_float('lambda_l1', 1e-3, 10, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
    }
    dtr = lgb.Dataset(Xtr, ytr, categorical_feature=cat_features)
    dva = lgb.Dataset(Xva, yva, reference=dtr, categorical_feature=cat_features)
    m = lgb.train(params, dtr, num_boost_round=2000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(80, verbose=False)])
    pred = m.predict(Xva_eval, num_iteration=m.best_iteration)
    return metrica_producto(df_va_201910, pred, 201912)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20, show_progress_bar=False)
print('mejor métrica (producto):', round(study.best_value, 4))
print('mejores params:', study.best_params)

## 13. Modelo final, predicción 202002 y submit a Kaggle
Reentrenamos con train+valid (usando `best_iteration` de la corrida de validación) y predecimos las filas de `periodo == 201912`. Sumamos por `product_id` y completamos los 780 productos a predecir.

In [ ]:
best_params = {
    'objective': 'regression_l1', 'metric': 'l1', 'verbosity': -1,
    'seed': PARAM['semilla'], 'bagging_freq': 1, **study.best_params,
}

# best_iteration desde la validación
dtr = lgb.Dataset(Xtr, ytr, categorical_feature=cat_features)
dva = lgb.Dataset(Xva, yva, reference=dtr, categorical_feature=cat_features)
m_tmp = lgb.train(best_params, dtr, num_boost_round=2000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(80, verbose=False)])
best_iter = m_tmp.best_iteration or 800
print('best_iteration:', best_iter)

# Reentreno final con train+valid
full_mask = df['target'].notna()
Xfull, yfull = df.loc[full_mask, features], df.loc[full_mask, 'target']
dfull = lgb.Dataset(Xfull, yfull, categorical_feature=cat_features)
modelo = lgb.train(best_params, dfull, num_boost_round=int(best_iter * 1.1))

In [ ]:
# Importancia de variables
imp = pd.DataFrame({'feature': features,
                    'gain': modelo.feature_importance('gain')}).sort_values('gain', ascending=False)
fig, ax = plt.subplots(figsize=(8, 8))
top = imp.head(25).iloc[::-1]
ax.barh(top['feature'], top['gain']); ax.set_title('LightGBM — importancia (gain)')
plt.tight_layout(); plt.show()
imp.head(15)

In [ ]:
# Predicción de 202002 (a partir de las filas de 201912) y agregación a producto
Xpred = df.loc[predict_mask, features]
pred  = np.clip(modelo.predict(Xpred), 0, None)

df_pred = df.loc[predict_mask, ['product_id', 'customer_id']].copy()
df_pred['tn'] = pred
sub = df_pred.groupby('product_id', as_index=False)['tn'].sum()

# Completar los 780 productos a predecir (los que no aparezcan -> 0)
sub = apredecir[['product_id']].merge(sub, on='product_id', how='left').fillna({'tn': 0.0})
sub = sub[['product_id', 'tn']]
print('submission:', sub.shape, '| tn total:', round(sub['tn'].sum(), 2))
sub.head()

In [ ]:
archivo = 'lgbm_imputnorm.csv'
sub.to_csv(archivo, index=False)
kaggle_submit(PARAM['kaggle_competition'], archivo,
              f"LGBM imput/norm p×c, racha0, cluster, optuna(best={round(study.best_value,4)})")
print('submit enviado:', archivo)

## Resumen de salidas generadas
- `panel_ceros.parquet` — panel con **ceros reales = 0** y **estructurales = NaN** + features de contexto.
- `parametros_normalizacion.csv` — `product_id, customer_id, mu, sd, inicio_serie, n_obs, flag_plano` (permite invertir `x = z·sd + mu`).
- `matz` / `prod_cluster` — series z-norm alineadas por edad y su cluster de forma (KShape).
- `df[features]` — dataset LightGBM (estructurales removidas, NaN preservados en lags, variables escaladas, máximo histórico, racha de ceros, cluster_id).
- `lgbm_imputnorm.csv` — submission a Kaggle (202002, agregada a producto).
